# Notebook B — Image‑based Liveness Evaluation (ONNX)
# -------------------------------------------------
# Evaluates: MiniFASNetV2.onnx, facebagnet_color_96.onnx
# Dataset structure:
# dataset/
# real/
# fake/
# -------------------------------------------------

In [1]:
import sys
print(sys.executable)

c:\Users\Leo\Documents\AI\Capstone\.venv\Scripts\python.exe


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import onnxruntime as ort
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from tqdm import tqdm

In [23]:
BASE_DIR = os.path.dirname(os.getcwd()) # notebook in app/notebooks/
MODELS_DIR = os.path.join(BASE_DIR, "models")
DATASET_DIR = os.path.join(BASE_DIR, "dataset")


MINIFASNET_PATH = os.path.join(MODELS_DIR, "MiniFASNetV2.onnx")
FACEBAGNET_PATH = os.path.join(MODELS_DIR, "facebagnet_color_96.onnx")


# MiniFASNet expects 80x80, FaceBagNet expects 96x96
IMAGE_SIZE_MINI = (80, 80)
IMAGE_SIZE_FACEBAG = (96, 96)
BATCH_SIZE = 32


print(f"Checking Models...\nMiniFASNet: {os.path.exists(MINIFASNET_PATH)}\nFaceBagNet: {os.path.exists(FACEBAGNET_PATH)}")


Checking Models...
MiniFASNet: True
FaceBagNet: True


In [36]:
def load_dataset(dataset_dir, image_size):
    X, y = [], []
    # Alphabetical order: 'fake' will be 0, 'real' will be 1
    for label in ["fake", "real"]:
        p = os.path.join(dataset_dir, label)
        if not os.path.isdir(p): 
            print(f"Warning: Directory {p} not found.")
            continue
        for f in os.listdir(p):
            img_path = os.path.join(p, f)
            img = cv2.imread(img_path)
            if img is None: continue
            
            # Models expect RGB, OpenCV loads BGR
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, image_size)
            X.append(img)
            y.append(label)
    return np.array(X), np.array(y)

print("Loading datasets...")
X_mini_raw, y = load_dataset(DATASET_DIR, IMAGE_SIZE_MINI)
X_face_raw, _ = load_dataset(DATASET_DIR, IMAGE_SIZE_FACEBAG)

le = LabelEncoder()
y_true = le.fit_transform(y) # fake=0, real=1

print(f"Total Samples: {len(X_mini_raw)}")
print(f"Classes: {le.classes_} -> mapped to [0, 1]")


Loading datasets...
Total Samples: 269
Classes: ['fake' 'real'] -> mapped to [0, 1]


In [45]:
def preprocess_mini(X):
    # MiniFASNetV2 usually uses 127.5 scale in 2026 production
    X = (X.astype(np.float32) - 127.5) / 128.0
    return np.transpose(X, (0, 3, 1, 2))


In [46]:
def preprocess_facebag(X):
    X = X.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    X = (X - mean) / std
    return np.transpose(X, (0, 3, 1, 2))

In [47]:
X_mini = preprocess_mini(X_mini_raw)
X_face = preprocess_facebag(X_face_raw)

In [82]:
session = ort.InferenceSession(MINIFASNET_PATH)
print(session.get_outputs()[0].shape)


['batch_size', 3]


In [92]:
def run_inference(model_path, X_processed, model_name="MiniFASNet"):
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    y_pred = []

    print(f"Running inference for {model_name}...")
    for i in tqdm(range(len(X_processed))):
        inp = np.expand_dims(X_processed[i], axis=0)
        output = session.run(None, {input_name: inp})[0][0]

        if model_name == "FaceBagNet":
            logits = session.run(None, {input_name: inp})[0][0]
            score_diff = logits[0] - logits[1]
            # Use the same denominator (250.0) as your backend
            normalized_spoof_score = 1.0 / (1.0 + np.exp(score_diff / 250.0))
            prediction = 1 if normalized_spoof_score > 0.90 else 0
        else:
            # Inside run_inference for MiniFASNet
            logits = session.run(None, {input_name: inp})[0][0]
            fake_logit = logits[0]
            real_logit = logits[1]
            # Use the same exp math from your backend
            exps = np.exp([fake_logit, real_logit])
            probs = exps / np.sum(exps)
            prediction = 1 if probs[1] > 0.50 else 0 # Adjust threshold to match backend

            # logits = session.run(None, {input_name: inp})[0][0] # Shape: (3,)
            
            # Apply Softmax across all 3 classes
            # exps = np.exp(logits - np.max(logits)) # Subtraction for numerical stability
            # probs = exps / np.sum(exps)
            
            # # Class 1 is usually 'Real' in MiniFASNet
            # # Prediction is 1 if index 1 has the highest probability
            # prediction = 1 if np.argmax(probs) == 1 else 0
            
            # Optional: If you want a specific threshold for "Realness"
            # prediction = 1 if probs[1] > 0.50 else 0
            # if i < 5:
            #     print(f"Sample {i} Probs: {probs} | Predicted Index: {np.argmax(probs)}")
            
        y_pred.append(prediction)
    return np.array(y_pred)

In [93]:
y_pred_mini = run_inference(MINIFASNET_PATH, X_mini, "MiniFASNet")
y_pred_face = run_inference(FACEBAGNET_PATH, X_face, "FaceBagNet")

Running inference for MiniFASNet...


100%|██████████| 269/269 [00:02<00:00, 112.34it/s]


Running inference for FaceBagNet...


100%|██████████| 269/269 [00:04<00:00, 57.95it/s]


In [94]:
def evaluate(name, y_true, y_pred):
    unique_labels = np.unique(np.concatenate([y_true, y_pred]))
    avg_mode = 'binary' if len(unique_labels) <= 2 else 'macro'
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=avg_mode, zero_division=0)
    rec = recall_score(y_true, y_pred, average=avg_mode, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n--- {name} Results ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    return [acc, prec, rec]

In [95]:
mini_metrics = evaluate("MiniFASNetV2", y_true, y_pred_mini)
face_metrics = evaluate("FaceBagNet", y_true, y_pred_face)


--- MiniFASNetV2 Results ---
Accuracy : 0.3494
Precision: 0.3494
Recall   : 1.0000
Confusion Matrix:
[[  0 175]
 [  0  94]]

--- FaceBagNet Results ---
Accuracy : 0.9219
Precision: 0.9398
Recall   : 0.8298
Confusion Matrix:
[[170   5]
 [ 16  78]]


In [96]:
results = pd.DataFrame([
    ["MiniFASNetV2", *mini_metrics],
    ["FaceBagNet", *face_metrics]
], columns=["Model", "Accuracy", "Precision", "Recall"])

display(results)

,Model,Accuracy,Precision,Recall
0,MiniFASNetV2,0.349442,0.349442,1.000000
1,FaceBagNet,0.921933,0.939759,0.829787


In [ ]:
results.to_csv(os.path.join(DATASET_DIR, "model_comparison_2026.csv"), index=False)
print("\nEvaluation complete. Results saved to dataset/model_comparison_2026.csv")

  0%|          | 0/9 [00:00<?, ?it/s]


InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: input for the following indices
 index: 0 Got: 32 Expected: 1
 Please fix either the inputs/outputs or the model.